# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [ ]:
# 1. Cấu hìnhfrom scripts.kaggle_config import setup_kaggle_env# ====== ABLATION MODE ======# Uncomment one of these for ablation study:BASE_YAML_CONFIG = "train/ablation/ablation_baseline_8000.yaml"# BASE_YAML_CONFIG = "train/ablation/ablation_tree_8000.yaml"# BASE_YAML_CONFIG = "train/ablation/ablation_edl_8000.yaml"# BASE_YAML_CONFIG = "train/ablation/ablation_counting_8000.yaml"# BASE_YAML_CONFIG = "train/ablation/ablation_full_8000.yaml"# ====== NORMAL MODE ======#BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"# Setup environmentRUN_UUID, env = setup_kaggle_env(    base_yaml_config=BASE_YAML_CONFIG,    notebook_path="uni-mumer-kaggle-dagshub v8.ipynb")

In [2]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /kaggle/working/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/unimumer

  added / updated specs:
    - python=3.10


The following packages will



==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

6e45162


Cloning into '/kaggle/working/test-unimer'...


In [4]:
%%bash
# 3.5 Download test data
set -e

cd "$PROJECT_DIR"

pip install -q gdown

gdown "1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L" -O crohme_test_images.zip

unzip -q crohme_test_images.zip -d "$PROJECT_DIR"

rm crohme_test_images.zip

Downloading...
From (original): https://drive.google.com/uc?id=1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L
From (redirected): https://drive.google.com/uc?id=1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L&confirm=t&uuid=4b737cac-3a0b-436a-bb9c-5d0869abfb5b
To: /kaggle/working/test-unimer/crohme_test_images.zip
100%|██████████| 1.63G/1.63G [00:29<00:00, 55.7MB/s]


In [8]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

Python 3.10.20
GPU: Tesla T4
MLflow: 3.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.45.0 requires starlette<1.0,>=0.40.0; sys_platform != "emscripten", but you have starlette 1.3.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prometheus-fastapi-instrumentator 8.0.2 requires starlette<2.0.0,>=1.0.0, but you have starlette 0.52.1 which is incompatible.


In [9]:
%%bash
# 4.5. Setup ablation dataset
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/setup_ablation_dataset.py --seed 42

📋 Detected ablation config: ablation_baseline_8000

✗ Missing: /kaggle/working/test-unimer/train/ablation_data/ablation_baseline_8000.parquet
🔄 Dataset invalid or missing, need rebuild

Creating ablation dataset: ablation_baseline_8000

Validating build...
✓ 8000 samples, schema OK
✅ Build successful

📊 Dataset ready:
   ablation_baseline_8000.parquet (61.55 MB)


2026-06-26 08:39:30,620 [INFO] ================================================================================
2026-06-26 08:39:30,620 [INFO] Creating Ablation Study Datasets
2026-06-26 08:39:30,620 [INFO] ================================================================================
2026-06-26 08:39:30,620 [INFO] Project dir: /kaggle/working/test-unimer
2026-06-26 08:39:30,621 [INFO] Output dir: /kaggle/working/test-unimer/train/ablation_data
2026-06-26 08:39:30,621 [INFO] Seed: 42
2026-06-26 08:39:30,621 [INFO] Dataset info: /kaggle/working/test-unimer/train/dataset_info.json
2026-06-26 08:39:30,621 [INFO] 
2026-06-26 08:39:30,621 [INFO] Loaded dataset_info.json with 133 datasets
2026-06-26 08:39:30,621 [INFO] Building single config: ablation_baseline_8000
2026-06-26 08:39:30,621 [INFO] ================================================================================
2026-06-26 08:39:30,621 [INFO] Creating ablation_baseline_8000
2026-06-26 08:39:30,621 [INFO] ======================

In [10]:
%%bash
# 5.5. Verify dataset registration
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/verify_ablation_registration.py

Verifying dataset registration for current config
📋 Current config: ablation_baseline_8000

✓ ablation_baseline_8000 registered in dataset_info.json
    file_name: ablation_data/ablation_baseline_8000.parquet
    formatting: sharegpt
    columns: ['messages', 'images']

✅ Dataset registration verified - Ready to train


In [11]:
# 5. Runtime YAML Override (tùy chọn)
from scripts.runtime_yaml import prepare_runtime_yaml

USE_RUNTIME_YAML_OVERRIDE = True

# Check if using ablation config
IS_ABLATION = "ablation" in BASE_YAML_CONFIG

if IS_ABLATION:
    print("🔬 ABLATION MODE DETECTED")
    print("⚠️  DO NOT override 'dataset' or 'max_samples'!")
    print("✅ Only override hyperparameters for quick testing\n")
    
    # ABLATION MODE: Only override hyperparameters
    # DO NOT override dataset or max_samples!
    YAML_OVERRIDES = {
        "num_train_epochs": 3,  # Quick test (original: 3)
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 64,
        "learning_rate": 1.0e-4,
        "logging_steps": 1,
        "save_steps": 23,
        "eval_steps": 23,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        # NO dataset override!
        # NO max_samples override!
        
        # Keep these None to use YAML defaults
        "cutoff_len": None,
        "val_size": None,
        "per_device_eval_batch_size": None,
        "save_total_limit": None,
        "lora_alpha": None,
        "lora_dropout": None,
        "warmup_ratio": None,
        "quantization_bit": None,
        "preprocessing_num_workers": None,
        "dataloader_num_workers": None,
        "bf16": None,
        "fp16": None,
    }
else:
    print("📝 NORMAL MODE")
    print("✅ Can override dataset and max_samples\n")
    
    # NORMAL MODE: Can override dataset
    YAML_OVERRIDES = {
        "dataset": "parquet_crohme_train",
        "max_samples": 3,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 64,
        "learning_rate": 1.0e-4,
        "lora_rank": 64,
        "logging_steps": 1,
        "save_steps": 23,
        "eval_steps": 23,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "output_dir": "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora",
        
        # Để None nếu không muốn override
        "cutoff_len": None,
        "val_size": None,
        "per_device_eval_batch_size": None,
        "lora_alpha": None,
        "lora_dropout": None,
        "warmup_ratio": None,
        "quantization_bit": None,
        "preprocessing_num_workers": None,
        "dataloader_num_workers": None,
        "bf16": None,
        "fp16": None,
    }

YAML_CONFIG, OUTPUT_DIR, YAML_DATA = prepare_runtime_yaml(
    project_dir=PROJECT_DIR,
    base_yaml_config=BASE_YAML_CONFIG,
    runtime_yaml_config=RUNTIME_YAML_CONFIG,
    use_override=USE_RUNTIME_YAML_OVERRIDE,
    overrides=YAML_OVERRIDES,
    strict_keys=True,
)

# Cập nhật biến môi trường
mlflow_tags = json.loads(os.environ["MLFLOW_TAGS"])
mlflow_tags.update({
    "dataset": str(YAML_DATA.get("dataset", "")),
    "yaml_config": YAML_CONFIG,
    "yaml_override": str(USE_RUNTIME_YAML_OVERRIDE).lower(),
    "ablation_mode": str(IS_ABLATION).lower(),
})

os.environ.update({
    "YAML_CONFIG": YAML_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MLFLOW_TAGS": json.dumps(mlflow_tags),
})

print(f"\n📊 Final Config:")
print(f"   Dataset: {YAML_DATA.get('dataset', 'N/A')}")
print(f"   Epochs: {YAML_DATA.get('num_train_epochs', 'N/A')}")
print(f"   Save/Eval Steps: {YAML_DATA.get('save_steps', 'N/A')}/{YAML_DATA.get('eval_steps', 'N/A')}")
print(f"   Save Total Limit: {YAML_DATA.get('save_total_limit', 'N/A')}")
print(f"   Output: {OUTPUT_DIR}")

🔬 ABLATION MODE DETECTED
⚠️  DO NOT override 'dataset' or 'max_samples'!
✅ Only override hyperparameters for quick testing

Runtime YAML Override: ON
YAML gốc: /kaggle/working/test-unimer/train/ablation/ablation_baseline_8000.yaml
YAML dùng để train: /kaggle/working/runtime_Uni-MuMER-train.yaml
Các key đã đổi:
  logging_steps: 10 -> 1

📊 Final Config:
   Dataset: ablation_baseline_8000
   Epochs: 3
   Save/Eval Steps: 23/23
   Save Total Limit: 2
   Output: saves/ablation/baseline_8000


In [12]:
%%bash
# 6. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

🏃 View run serious-dog-821 at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/3c2cf516959a4845a0a408c6511c1d1d
🧪 View experiment at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1
DagsHub MLflow connection OK: https://dagshub.com/NhatPot/test-unimer.mlflow


In [ ]:
%%bash
# 7. Training
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
MPLBACKEND=Agg llamafactory-cli train "$YAML_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

In [ ]:
%%bash# 8. Upload trained model lên DagsHubset -euo pipefailcd "$PROJECT_DIR"source "$CONDA_DIR/bin/activate" unimumerecho "============================================================"echo "           UPLOAD TRAINED MODEL TO DAGSHUB"echo "============================================================"# Tìm checkpoint cuối cùngLAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)if [[ -z "$LAST_CKPT" ]]; then  echo "ERROR: No checkpoint found in $OUTPUT_DIR"  exit 1fiecho "Latest checkpoint: $LAST_CKPT"echo ""# Upload model checkpoint và configpython scripts/dagshub_logger.py upload   --experiment "$MLFLOW_EXPERIMENT_NAME"   --run-uuid "$RUN_UUID"   --config "$YAML_CONFIG"   --output-dir "$OUTPUT_DIR"   --project-dir "$PROJECT_DIR"   --notebook "$NOTEBOOK_PATH"echo ""echo "============================================================"echo "✅ Model uploaded to DagsHub"echo "============================================================"echo ""echo "📥 Download và test ở máy local:"echo "   1. Vào: https://dagshub.com/NhatPot/test-unimer.mlflow"echo "   2. Tìm run: $RUN_UUID"echo "   3. Download artifacts/checkpoints"echo "   4. Test local với checkpoint đã download"echo "============================================================"